## Document Loading & Chunk Splitting

In [ ]:
from rag.utils.pdf import CustomPDFLoader

In [ ]:
loader = CustomPDFLoader("https://www.nibe.eu/assets/documents/16900/231844-5.pdf")
transformed_docs = loader.load(chunk_docs=True)

In [ ]:
# Examine chunk lengths
[len(i.page_content) for i in transformed_docs]

In [ ]:
len(transformed_docs)

## Using LCEL-based chains and Langfuse callback

In [ ]:
# These are the same chain, the hp_installer_bot_chain import path is interfaced through the rag module
from rag import hp_installer_bot_chain
from rag.chains import rag_chain_with_source

In [ ]:
rag_chain_with_source.get_graph().draw_png("chain_dag.png")

In [ ]:
from rag.utils.callbacks import langfuse_handler_from_config

# Check connection to Langfuse host
langfuse_handler = langfuse_handler_from_config()
langfuse_handler.auth_check()

In [ ]:
hp_installer_bot_chain.invoke("What is your name?", config={"callbacks": [langfuse_handler]})

In [ ]:
result = rag_chain_with_source.invoke("What information do you have to hand? Which installations guides do you have?", config={"callbacks": [langfuse_handler]})
# result is a dictionary with the keys: context, query, answer
# the sources used can be post processed
# the chatbot system prompt can also be modified
result

In [ ]:
rag_chain_with_source.invoke("What is your name?", config={"callbacks": [langfuse_handler]})

In [ ]:
config1 = rag_chain_with_source.with_config(configurable={"temperature": 0.3})
config1.invoke("What is your name?", config={"callbacks": [langfuse_handler]})
config1.invoke("What information do you have to hand? Which installations guides do you have?", config={"callbacks": [langfuse_handler]})

In [ ]:
config2 = rag_chain_with_source.with_config(configurable={"model_name": "gpt-4-turbo","temperature": 0.3})
config2.invoke("What is your name?", config={"callbacks": [langfuse_handler]})
config2.invoke("What information do you have to hand? Which installations guides do you have?", config={"callbacks": [langfuse_handler]})

In [ ]:
config3 = rag_chain_with_source.with_config(configurable={"model_name": "gpt-4-turbo","temperature": 0.3})

In [ ]:
from rag.vector_databases.retrievers import nibe_manual_retriever

In [ ]:
# Documents can still be retrieved this way, but embeddings generation of the query is not logged.
nibe_manual_retriever.invoke("What is your name?", config={"callbacks": [langfuse_handler]})

In [ ]:
nibe_manual_retriever.get_graph().draw_png("retriever_dag.png")

In [ ]:
retriever_config1 = nibe_manual_retriever.with_config(configurable={
        "search_kwargs": {"score_threshold": 0.74},
        "search_type": "similarity_score_threshold",
    }
)
res = retriever_config1.invoke("What is your name?", config={"callbacks": [langfuse_handler]})
len(res) # no documents have similarity > 0.74

In [ ]:
retriever_config1 = nibe_manual_retriever.with_config(configurable={"search_kwargs": {"k": 7}})
res = retriever_config1.invoke("What is your name?", config={"callbacks": [langfuse_handler]})
len(res) # returns k=7 documents